# 아이오닉 5 PDF 소제목별 텍스트·이미지 청킹

- 수업에서 배운 `pypdf → Document → 청킹 → Pandas` 흐름에 이미지 저장과 연결을 추가합니다.
- ‘PDF 읽기 → 글과 그림 수집 → 소제목별 정리 → 청크 생성 → 임베딩 → 검색 저장소 구축

### 1. PDF 문서 청킹에 필요한 도구 불러오기


In [ ]:
# pathlib에서 Path를 가져옵니다. 파일명·확장자를 꺼내거나 폴더 경로를 연결하는 도구입니다.
from pathlib import Path
# 청크 정보를 JSON 문자열로 바꿔 저장하고, 저장된 JSON을 다시 읽는 도구입니다.
import json

# PDF를 열고 페이지별 글자와 삽입 이미지를 꺼내는 PdfReader를 가져옵니다.
from pypdf import PdfReader
# 본문(page_content)과 출처(metadata)를 한 묶음으로 담는 자료 형식입니다. Word 파일이 아닙니다.
from langchain_core.documents import Document
# 문단·줄바꿈 등의 경계를 고려하여 긴 글을 작은 텍스트 청크로 나누는 도구입니다.
from langchain_text_splitters import RecursiveCharacterTextSplitter
# display는 표·그림을 보여주고, Image는 저장된 그림 파일을 노트북에서 표시할 때 사용합니다.
from IPython.display import display, Image
# 표를 다루는 pandas를 pd라는 짧은 이름으로 사용합니다.
import pandas as pd
# Python이 우리 프로젝트의 모듈을 찾을 폴더를 추가합니다.
import sys
# car_search_rag 폴더 자체가 아니라, 그 폴더를 포함한 src를 지정합니다.
src_dir = Path("/Users/anna/mle-02-p1-team1/src")
# 다른 컴퓨터에서는 위 경로를 실제 프로젝트 위치에 맞게 수정합니다.
if not (src_dir / "car_search_rag" / "common" / "storage_manager.py").is_file():
    raise FileNotFoundError("src_dir 경로를 확인하세요. storage_manager.py를 찾을 수 없습니다.")
# sys.path는 Python이 import할 파일을 찾는 폴더 목록입니다. 중복 추가는 방지합니다.
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
# 검색 경로를 등록한 뒤 팀 공통 저장소 도구를 가져옵니다. 업로드는 아직 실행하지 않습니다.
from car_search_rag.common.storage_manager import StorageManager


### 2. PDF와 저장 폴더 준비


In [ ]:
# 읽을 PDF의 실제 위치를 문자열로 지정합니다. 다른 파일을 쓰면 이 경로를 바꿉니다.
pdf_path = "/Users/anna/Downloads/NE1_2027_ko_KR.pdf"
# 개인 작업 폴더를 Path 객체로 지정합니다. 현재 컴퓨터의 경로이므로 다른 환경에서는 수정합니다.
work_dir = Path("/Users/anna/mle-02-p1-team1/src/car_search_rag/anna_rag")
# Path의 /는 폴더 연결입니다. .stem은 확장자를 뺀 PDF 이름이며, PDF별 출력 폴더를 구분합니다.
output_dir = work_dir / "data" / "manual_chunks" / Path(pdf_path).stem
# 실제 폴더를 만듭니다. parents=True는 중간 폴더도 생성, exist_ok=True는 기존 폴더를 허용합니다.
output_dir.mkdir(parents=True, exist_ok=True)

# 지정한 PDF를 읽는 객체를 reader에 저장합니다. 아직 청킹하거나 AI에 전달하지 않습니다.
reader = PdfReader(pdf_path)
# reader.pages는 페이지 모음이고, len()은 개수를 셉니다. print()는 결과를 화면에 보여줍니다.
print("전체 페이지 수:", len(reader.pages))
# 이후 이미지와 JSON이 저장될 기준 폴더를 확인합니다. 이 줄 자체는 파일을 저장하지 않습니다.
print("저장 위치:", output_dir)


### 3. 특정 PDF 페이지의 텍스트 불러오기

In [ ]:
# PDF 뷰어에서 보고 싶은 페이지 번호를 직접 정합니다. 여기서는 18번째 페이지입니다.
target_page = 18
# 원본에 인쇄된 쪽수를 직접 기록합니다. PDF 페이지 순서와 다르며 표시용 문자열로 보관합니다.
printed_page = "10"  # 원본 페이지에 인쇄된 쪽수입니다.
# Python 목록은 0부터 세므로 18번째 페이지를 17번 위치에서 가져옵니다.
page = reader.pages[target_page - 1]
# 페이지의 글자를 추출합니다. 결과가 None이나 빈 값이면 or 오른쪽의 빈 문자열을 사용합니다.
text = page.extract_text() or ""

# 추출된 페이지 전체 글을 출력합니다. 화면과 비교해 제목이나 깨진 글자를 확인하세요.
print(text)


### 4. 소제목의 텍스트 선택·정리

시작 제목부터 다음 제목 직전까지 선택합니다. 다음 제목이 다른 페이지에 있는 경우에는 먼저 페이지를 연결하는 별도 처리가 필요합니다.

세 가지 깨진 표기는 이 PDF의 원본과 비교한 값입니다. 모든 PDF에 적용되는 규칙은 아닙니다.


In [ ]:
# 가져올 소제목의 시작 문구를 직접 지정합니다. 코드는 소제목을 자동으로 인식하지 않습니다.
title = "비상시 충전 커넥터 분리하기"
# 다음 소제목을 끝 경계로 지정합니다. 이 제목의 직전까지 가져옵니다.
next_title = "완속 충전기 사용하기"

# 소제목 묶음의 이름표입니다. 같은 본문과 그림을 연결하며, 다른 소제목에는 다른 ID를 사용합니다.
parent_id = "emergency_connector_p18"

# 앞 셀에서 추출한 text 값을 page_text라는 이름으로도 사용합니다. 다시 추출하는 명령은 아닙니다.
page_text = text

# .index()는 지정한 제목이 시작하는 문자 위치를 찾습니다. 제목이 없으면 ValueError가 납니다.
start = page_text.index(title)
# 현재 제목의 끝(start + 제목 길이) 이후부터 다음 제목을 찾아 끝 위치로 저장합니다.
end = page_text.index(next_title, start + len(title))
# [start:end]는 시작 위치를 포함하고 끝 위치는 제외하여 글을 자르는 슬라이싱입니다.
raw_section_text = page_text[start:end]
# .strip()으로 맨 앞뒤의 공백·줄바꿈만 제거합니다. raw_section_text에는 정리 전 글이 남습니다.
section_text = raw_section_text.strip()

# 원본에서 확인한 깨진 경고 표기를 치환하고 결과를 다시 저장합니다. AI가 뜻을 판단하지 않습니다.
section_text = section_text.replace("҃Ҋ", "[경고]")
# 이 PDF에서 확인한 주의 표기를 바꿉니다. 다른 PDF에도 같은 규칙을 무조건 적용하지 않습니다.
section_text = section_text.replace("઱੄", "[주의]")
# 확인된 알아두기 표기를 바꿉니다. 그림이나 표 구조를 복원하는 기능은 아닙니다.
section_text = section_text.replace("ӝ", "[알아두기]")
# 선택하고 정리한 소제목의 본문을 확인합니다. 다른 설명이 섞이거나 경고가 빠지지 않았는지 봅니다.
print(section_text)


## 5. 페이지의 이미지 미리보기

이미지 번호는 PDF의 저장 순서이며, 위에서 아래로 보이는 순서와 다를 수 있습니다. 원본 페이지와 미리보기를 비교해 해당 소제목의 그림만 고르세요.

`page.images`는 PDF에 삽입된 이미지 객체를 꺼냅니다. 벡터로 그려진 그림·아이콘은 빠질 수 있고, 이미지 속 글자를 읽는 OCR은 수행하지 않습니다. 이번 페이지에서는 큰 설명 그림 1장을 추출하며 경고·주의 아이콘은 포함하지 않습니다.


In [ ]:
# 이 페이지의 삽입 이미지들을 리스트에 담습니다. 소제목 범위로 자동 선택하지 않으며 벡터 그림은 빠질 수 있습니다.
page_images = list(page.images)
# 이미지 목록의 길이를 출력합니다. 원본 화면의 모든 그림·아이콘 개수와 같지는 않을 수 있습니다.
print("추출 가능한 이미지 수:", len(page_images))

# 이미지를 한 장씩 꺼냅니다. enumerate는 1부터 붙인 표시 번호와 이미지 객체를 함께 줍니다.
for image_no, pdf_image in enumerate(page_images, start=1):
    # 선택에 쓸 번호와 추출 이미지의 이름을 보여줍니다. 본문의 2C_... 식별 문자열과는 다릅니다.
    print("이미지 번호:", image_no, "원본 이름:", pdf_image.name)
    # Pillow 이미지 객체를 RGB 색상으로 변환해 미리보기 변수에 담습니다. 그림 내용을 해석하지는 않습니다.
    preview = pdf_image.image.convert("RGB")
    # 현재 그림을 노트북 화면에 보여줍니다. 파일 저장은 뒤의 7번 단계에서 합니다.
    display(preview)

# 이미지 목록이 비어 있을 때만 아래 안내를 실행합니다. not은 여기서 빈 목록인지 확인합니다.
if not page_images:
    # 추출 이미지가 0개임을 알립니다. PDF 화면에 선·도형으로 그린 그림이 있는지는 별도 확인이 필요합니다.
    print("삽입 이미지가 없습니다. 벡터 그림 유무는 원본 PDF에서 확인하세요.")


## 6. 소제목에 해당하는 이미지 선택

번호를 직접 선택하는 이유는 같은 페이지에 다른 소제목의 이미지가 있을 수 있기 때문입니다. 이미지 설명은 원본과 주변 본문을 보고 직접 작성합니다. AI가 그림을 해석한 결과가 아닙니다.

18페이지의 1번 그림을 확인하여 아래에 연결했습니다. 여러 그림이면 `2: "두 번째 그림 설명"`처럼 추가합니다. 이미지가 없는 소제목이면 `{}`로 둡니다.


In [ ]:
# 사용할 이미지 번호: 설명을 딕셔너리로 지정합니다. 사람이 고른 그림만 처리하며 없으면 {}로 둡니다.
selected_image_captions = {
    # 미리보기의 1번 그림에 직접 작성한 설명을 붙입니다. 나중에 이 설명을 텍스트 임베딩에 사용합니다.
    1: "화물칸의 충전 커넥터 비상 해제 케이블을 확대해 보여주는 그림"
}

# .items()로 선택 딕셔너리에서 번호(image_no)와 설명(caption)을 한 쌍씩 꺼냅니다.
for image_no, caption in selected_image_captions.items():
    # 번호가 1보다 작거나 이미지 수보다 크면 잘못된 선택입니다. or는 둘 중 하나라도 참이면 됩니다.
    if image_no < 1 or image_no > len(page_images):
        # 잘못된 번호를 메시지에 넣고 실행을 멈춥니다. f 문자열의 {변수}는 실제 값으로 바뀝니다.
        raise ValueError(f"이미지 {image_no}번이 없습니다. 5번 미리보기를 확인하세요.")
    # 설명 앞뒤 공백을 제거했을 때 빈 문자열이면 검사에 걸립니다. 공백만 적어도 빈 설명입니다.
    if not caption.strip():
        # 검색할 설명이 없으므로 안내와 함께 멈춥니다. 설명의 내용이 정확한지까지 검사하지는 않습니다.
        raise ValueError("검색에 사용할 이미지 설명을 입력하세요.")


## 7. 이미지 파일과 이미지 청크 만들기

원본 파일과 화면 표시용 PNG를 함께 저장합니다. PNG에는 해당 이미지 안의 글자·선이 보존되지만, 이미지 바깥에 따로 그려진 표식은 포함되지 않을 수 있습니다.

`page_content`에는 검색용 제목·설명을 넣고, `metadata`에는 그림 경로와 출처를 넣습니다. `image_path`는 `output_dir` 기준 상대 경로입니다.


In [ ]:
# 확장자를 뺀 PDF 이름을 문서 ID로 사용합니다. 파일 내용의 변경을 자동 감지하는 ID는 아닙니다.
source_id = Path(pdf_path).stem
# 저장 기준 폴더 아래에 이번 소제목의 이미지 폴더 경로를 만듭니다.
section_dir = output_dir / parent_id
# 소제목 폴더를 실제로 생성합니다. 이미 있는 폴더도 사용할 수 있습니다.
section_dir.mkdir(parents=True, exist_ok=True)

# 본문과 이미지가 공유할 출처 정보를 딕셔너리로 모읍니다. metadata는 자료를 설명하는 정보입니다.
common_metadata = {
    # .name으로 확장자까지 포함한 원본 파일명을 기록합니다.
    "source": Path(pdf_path).name,
    # 어떤 PDF에서 가져왔는지 구분하는 ID입니다.
    "source_id": source_id,
    # 차종을 직접 기록합니다. 나중에 이 항목으로 검색 필터를 구현해야 차종을 분리할 수 있습니다.
    "vehicle_id": "ioniq5",
    # PDF 뷰어 기준 페이지를 출처로 기록합니다.
    "pdf_page": target_page,
    # 원본에 인쇄된 쪽수를 기록합니다. 자동 계산되는 값이 아닙니다.
    "printed_page": printed_page,
    # 왼쪽은 항목 이름, 오른쪽은 앞서 지정한 소제목 변수의 값입니다.
    "title": title,
    # 같은 소제목의 전체 설명·텍스트·이미지를 이어줄 공통 ID입니다.
    "parent_id": parent_id
}

# 이미지 청크를 담을 빈 리스트를 만듭니다. 셀을 다시 실행하면 이 목록도 새로 시작합니다.
image_chunks = []
# .items()로 선택 딕셔너리에서 번호(image_no)와 설명(caption)을 한 쌍씩 꺼냅니다.
for image_no, caption in selected_image_captions.items():
    # 표시 번호 1번은 리스트의 0번 위치이므로 1을 빼고 실제 이미지를 꺼냅니다.
    pdf_image = page_images[image_no - 1]
    # 문서·소제목·번호를 합쳐 그림 ID를 만듭니다. :02d는 정수 1을 01처럼 두 자리로 표시합니다.
    image_id = f"{source_id}_{parent_id}_image_{image_no:02d}"

    # 이미지 이름에서 .jpg 같은 확장자를 가져옵니다. 파일명 전체를 가져오는 .name과 다릅니다.
    original_suffix = Path(pdf_image.name).suffix
    # 추출 원본 이미지의 저장 경로를 정합니다. 이 줄만으로 파일을 저장하지는 않습니다.
    original_path = section_dir / f"{image_id}_original{original_suffix}"
    # 추출한 이미지의 바이트 데이터를 실제 파일로 씁니다. 같은 경로의 파일이 있으면 덮어씁니다.
    original_path.write_bytes(pdf_image.data)

    # 화면 표시용 PNG의 저장 경로를 정합니다.
    image_path = section_dir / f"{image_id}.png"
    # CMYK 그림도 PNG로 저장할 수 있도록 RGB로 변환한 뒤 파일로 저장합니다. 원본 PDF는 바꾸지 않습니다.
    pdf_image.image.convert("RGB").save(image_path)

    # 공통 출처 딕셔너리를 복사합니다. 이미지 전용 항목을 추가해도 공통 딕셔너리가 바뀌지 않게 합니다.
    metadata = common_metadata.copy()
    # 복사한 딕셔너리에 이번 이미지 전용 항목들을 추가합니다. 같은 키가 있으면 값을 갱신합니다.
    metadata.update({
        # 이 이미지 청크를 구분하는 고유 ID입니다.
        "chunk_id": image_id,
        # 이미지 설명 청크라는 표시입니다. 이 값만으로 AI가 그림을 이해하는 것은 아닙니다.
        "chunk_type": "image",
        # 미리보기에서 선택한 번호를 기록합니다.
        "image_no": image_no,
        # 기준 폴더를 뺀 상대 경로를 / 구분자의 문자열로 저장합니다. 다시 읽을 때 output_dir과 합칩니다.
        "image_path": image_path.relative_to(output_dir).as_posix(),
        # 추출 원본 파일도 상대 경로로 기록합니다. 폴더를 옮길 때 JSON과 실제 그림을 함께 옮겨야 합니다.
        "original_image_path": original_path.relative_to(output_dir).as_posix(),
        # 캡션을 사람이 직접 작성했다는 표시입니다. 현재 코드에는 이미지 자동 해석이 없습니다.
        "caption_source": "manual"
    })
    # 이미지 한 장의 검색용 설명과 출처를 하나의 Document로 묶습니다.
    image_doc = Document(
        # 검색할 텍스트를 구성합니다. \n은 줄바꿈이며, 여기에 이미지 픽셀 자체는 넣지 않습니다.
        page_content=f"소제목: {title}\n이미지 설명: {caption}",
        # Document의 metadata 인수에 앞에서 만든 출처·이미지 경로 딕셔너리를 넣습니다.
        metadata=metadata
    )
    # 완성한 이미지 청크 한 개를 결과 리스트 끝에 추가합니다.
    image_chunks.append(image_doc)

# 완성된 이미지 청크의 개수를 출력합니다.
print("이미지 청크 수:", len(image_chunks))


## 8. 전체 설명 Document 만들기

`parent_doc`에는 경고·주의를 포함한 전체 설명을 보관합니다. 이미지 청크나 작은 텍스트 청크가 검색되면 `parent_id`로 이 설명을 함께 가져옵니다.


In [ ]:
# 청킹 전의 전체 설명을 보관합니다. 작은 청크 검색 후 경고·조건을 함께 확인할 자료입니다.
parent_doc = Document(
    # 정리한 소제목의 전체 본문을 넣습니다.
    page_content=section_text,
    # 전체 설명에도 같은 출처와 parent_id를 붙여 작은 청크·그림과 연결합니다.
    metadata=common_metadata.copy()
)
# Document 한 개를 리스트로 감쌉니다. 다음 분할 도구가 Document 목록을 입력받기 때문입니다.
pdf_docs = [parent_doc]
# 전체 설명의 문서명·페이지·소제목 ID가 맞는지 화면에서 확인합니다.
print(parent_doc)



## 9. 텍스트 청킹 후 이미지 청크와 합치기

텍스트만 `split_documents()`로 나눕니다. 이미지는 한 장이 한 청크이므로 분할하지 않습니다. `400/80`은 수업에서 사용한 출발값이며, 이미지 해상도와 관계없는 글자 수 설정입니다.


In [ ]:
# 텍스트 분할 도구의 기준을 정합니다. 실제 분할은 아래 split_documents 호출에서 실행됩니다.
splitter = RecursiveCharacterTextSplitter(
    # 기본 문자 수 기준으로 최대 400자 규모의 청크를 만듭니다. 토큰 수나 이미지 크기가 아닙니다.
    chunk_size=400,
    # 앞뒤 문맥을 일부 겹치게 합니다. 분할 경계에 따라 실제 겹침은 80자보다 적거나 없을 수 있습니다.
    chunk_overlap=80
)
# 전체 설명을 텍스트 청크로 나눕니다. 원래 메타데이터도 각 청크에 전달됩니다.
chunks = splitter.split_documents(pdf_docs)

# 텍스트 청크를 하나씩 꺼내면서 1부터 순서를 붙입니다. i는 번호, chunk는 현재 문서입니다.
for i, chunk in enumerate(chunks, start=1):
    # 이 소제목 안에서 몇 번째 텍스트 청크인지 기록합니다.
    chunk.metadata["chunk_no"] = i
    # 텍스트 청크라는 종류 표시를 추가합니다.
    chunk.metadata["chunk_type"] = "text"
    # 문서·소제목·순서를 합쳐 개별 텍스트 청크의 ID를 만듭니다.
    chunk.metadata["chunk_id"] = f"{source_id}_{parent_id}_text_{i:02d}"

# 두 리스트를 연결합니다. 텍스트 청크 뒤에 이미지 청크가 오며, 그림을 추가 분할하지는 않습니다.
all_chunks = chunks + image_chunks
# 텍스트를 나눈 결과가 몇 개인지 확인합니다.
print("텍스트 청크:", len(chunks))
# 선택한 이미지 청크가 몇 개인지 확인합니다.
print("이미지 청크:", len(image_chunks))
# 텍스트와 이미지 청크를 합친 전체 개수를 확인합니다.
print("전체 청크:", len(all_chunks))


## 10. Pandas로 결과 확인

한 행이 텍스트 또는 이미지 청크 하나입니다. 텍스트 행에는 이미지 경로가 없어 빈 문자열을 넣습니다.


In [ ]:
# 표의 각 행을 담을 빈 리스트입니다. 한 청크가 표의 한 행이 됩니다.
rows = []
# 텍스트와 이미지 청크를 하나씩 꺼내 필요한 데이터를 읽습니다.
for chunk in all_chunks:
    # 이번 청크의 정보를 딕셔너리 한 개로 만들어 표의 행 목록에 추가합니다.
    rows.append({
        # 왼쪽 한글은 표의 열 이름이고, 오른쪽은 해당 청크의 ID를 읽는 코드입니다.
        "청크ID": chunk.metadata["chunk_id"],
        # text인지 image인지 표에 표시합니다.
        "종류": chunk.metadata["chunk_type"],
        # 해당 청크가 속한 소제목을 표시합니다.
        "소제목": chunk.metadata["title"],
        # 원본에서 확인할 PDF 페이지를 표시합니다.
        "PDF페이지": chunk.metadata["pdf_page"],
        # 전체 설명과 그림을 찾을 연결 ID를 표시합니다.
        "전체설명ID": chunk.metadata["parent_id"],
        # 텍스트 청크이면 본문, 이미지 청크이면 제목과 캡션을 표시합니다.
        "내용": chunk.page_content,
        # .get()은 키가 없으면 기본값을 반환합니다. 텍스트 청크에는 이미지 경로 대신 빈 문자열을 넣습니다.
        "이미지경로": chunk.metadata.get("image_path", "")
    })

# 딕셔너리 행 목록을 Pandas 표로 바꿉니다. 파일 저장이나 DB 적재는 아닙니다.
chunks_df = pd.DataFrame(rows)
# .shape는 (행 개수, 열 개수)입니다. 이번 기본 예시는 청크 3개의 3행 7열입니다.
print("행·열 수:", chunks_df.shape)
# 열마다 숫자인지 문자열인지 등의 자료형을 확인합니다.
print(chunks_df.dtypes)
# 다음에 출력할 결과가 결측치 개수임을 알리는 안내 문구입니다.
print("결측치 개수:")
# 비어 있는 값(None 등)을 열별로 셉니다. 빈 문자열 ""은 이 검사에서 결측치로 세지 않습니다.
print(chunks_df.isna().sum())
# 표를 노트북에 보여줍니다. 본문 누락·잘못된 페이지·그림 연결은 직접 확인해야 합니다.
display(chunks_df)


## 11. 검색된 청크의 전체 설명과 관련 그림 가져오기

아직 검색기를 만들지 않았으므로 첫 텍스트 청크가 검색되었다고 가정합니다. `selected_chunk = image_chunks[0]`으로 바꾸면 이미지 청크에서 전체 설명을 찾는 흐름도 확인할 수 있습니다.

연결된 그림은 **같은 소제목 단위의 참고 자료**입니다. 각각의 짧은 텍스트 청크와 그림이 일대일로 대응한다는 뜻은 아닙니다.


In [ ]:
# 첫 청크가 검색됐다고 가정합니다. 실제 질문 검색이 아니라 연결 구조를 확인하는 예시입니다.
selected_chunk = all_chunks[0]
# 선택한 청크가 속한 소제목의 ID를 꺼냅니다.
selected_parent_id = selected_chunk.metadata["parent_id"]
# 선택한 청크의 원본 문서 ID도 꺼내 다른 문서의 같은 이름과 혼동하지 않게 합니다.
selected_source_id = selected_chunk.metadata["source_id"]

# 보관한 전체 설명 문서를 하나씩 살펴봅니다. 현재 예시는 한 문서만 들어 있습니다.
for doc in pdf_docs:
    # ==는 값이 같은지 비교합니다. 먼저 소제목 ID가 일치하는지 확인합니다.
    if (doc.metadata["parent_id"] == selected_parent_id
            # and이므로 문서 ID까지 같아야 아래 전체 설명을 출력합니다.
            and doc.metadata["source_id"] == selected_source_id):
        # 짧은 청크만이 아니라 경고·주의를 포함한 전체 설명을 출력합니다.
        print(doc.page_content)
        # 원본 문서명과 페이지를 보여줍니다. PDF를 자동으로 여는 명령은 아닙니다.
        print("출처:", doc.metadata["source"], "PDF", doc.metadata["pdf_page"], "페이지")

# 이미지 청크들을 하나씩 살펴보며 같은 소제목의 그림을 찾습니다.
for image_doc in image_chunks:
    # 이미지가 속한 소제목 ID가 선택한 청크의 소제목 ID와 같은지 검사합니다.
    if (image_doc.metadata["parent_id"] == selected_parent_id
            # 이미지의 문서 ID도 같아야 합니다. 두 조건을 모두 만족하는 그림만 표시합니다.
            and image_doc.metadata["source_id"] == selected_source_id):
        # 관련 이미지의 제목과 설명을 화면에 출력합니다.
        print(image_doc.page_content)
        # 기준 폴더와 상대 경로를 합칩니다. str은 경로를 문자열로 바꾸고, Image와 display가 그림을 보여줍니다.
        display(Image(filename=str(output_dir / image_doc.metadata["image_path"])))


## 12. 다음 실습에서 사용할 JSON 저장

전체 설명, 텍스트 청크, 이미지 청크를 저장합니다. 이미지 파일은 별도로 보관하므로 다른 컴퓨터로 옮길 때는 `output_dir` 폴더 전체를 복사하세요. 같은 소제목을 다시 실행하면 같은 JSON 파일을 갱신합니다. 다른 소제목은 다른 `parent_id`를 사용하세요.

이미지 선택을 바꾸면 이전 이미지 파일이 폴더에 남을 수 있습니다. 다음 단계에서는 폴더 전체를 적재하지 말고, 이 JSON에 등록된 청크만 읽으세요.


In [ ]:
# JSON으로 저장할 청크 데이터를 모을 빈 리스트를 만듭니다.
records = []
# 텍스트와 이미지 청크를 하나씩 꺼내 필요한 데이터를 읽습니다.
for chunk in all_chunks:
    # Document의 필요한 내용을 JSON으로 표현 가능한 일반 딕셔너리로 바꿔 목록에 넣습니다.
    records.append({
        # 검색할 본문 또는 이미지 캡션을 문자열로 보관합니다.
        "page_content": chunk.page_content,
        # 해당 청크의 출처·ID·파일 경로를 함께 보관합니다. 이미지 파일 자체는 넣지 않습니다.
        "metadata": chunk.metadata
    })

# 최종 저장 내용을 전체 설명(parent)과 청크 목록(chunks)으로 묶습니다.
result = {
    # 작게 나누기 전 소제목 전체 설명을 별도 항목으로 남깁니다.
    "parent": {
        # 경고·주의가 포함된 전체 본문을 기록합니다.
        "page_content": parent_doc.page_content,
        # 전체 설명의 출처와 연결 ID를 기록합니다.
        "metadata": parent_doc.metadata
    },
    # 위 반복문에서 만든 텍스트·이미지 청크 목록을 넣습니다.
    "chunks": records
}
# 소제목 ID를 JSON 파일명으로 사용합니다. 같은 ID로 저장하면 기존 파일을 갱신합니다.
json_path = output_dir / f"{parent_id}.json"
# dumps로 JSON 문자열을 만들고 write_text로 저장합니다. 한글 유지(False), 두 칸 들여쓰기, UTF-8 인코딩을 지정합니다.
json_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
# 실제로 저장한 JSON 파일의 위치를 출력합니다.
print("저장 완료:", json_path)

# 저장 파일을 문자열로 읽고 loads로 Python 딕셔너리·리스트 구조로 되돌립니다.
saved = json.loads(json_path.read_text(encoding="utf-8"))
# 재읽은 청크 목록의 개수를 확인합니다. 그림을 다시 쓰려면 JSON뿐 아니라 이미지 파일도 필요합니다.
print("저장된 청크 수:", len(saved["chunks"]))


## 이번 단계의 결과

- `pdf_docs`: 경고·주의를 포함한 소제목 전체 설명
- `chunks`: 작은 텍스트 청크
- `image_chunks`: 그림의 설명·파일 경로를 가진 이미지 청크
- `all_chunks`, `chunks_df`: 텍스트와 이미지 청크를 합친 결과
- `data/manual_chunks/...`: 원본 이미지, 표시용 PNG, 청크 JSON

**다음 단계:** `all_chunks`의 `page_content`를 임베딩하고 메타데이터와 함께 DB에 저장합니다. 검색 결과의 `parent_id`로 전체 설명과 그림을 연결합니다.

이 노트북은 한 소제목의 실습입니다. 전체 PDF 자동 소제목 분류, 이미지 자동 해석, OCR, 벡터 그림 추출, 임베딩·DB 저장은 아직 수행하지 않습니다.

참고: [pypdf 공식 이미지 추출 문서](https://pypdf.readthedocs.io/en/stable/user/extract-images.html)
